In [1]:
"""
Wczytuje surowa baze odcinkow pomiarowych GPR 2025 (drogi krajowe +
wojewodzkie) z oficjalnych plikow GDDKiA/ZDW - geometria (Tab01) polaczona
z natezeniem ruchu (Tab02) po wspolnym kluczu "Numer odcinka pomiarowego".

Rozne struktury kolumn dla DK i DW (inna liczba kolumn w naglowku), oraz
rozne formaty liczb (DK: liczby, DW: tekst z przecinkiem dziesietnym) -
obsluzone osobno dla kazdego typu drogi.

Wymaga: openpyxl, cztery pliki GPR 2025 (Tab01/Tab02 x dk/dw)
Wynik: funkcja wczytaj_baze_segmentow() zwracajaca DataFrame z kolumnami:
       odcinek_id, droga, typ_drogi (dk/dw), lat, lon, sdrr_ogolem,
       sam_osobowe, lekkie_ciezarowe, ten_t (tylko dk)
"""

import openpyxl
import pandas as pd


def polska_liczba(wartosc):
    """Konwertuje polska notacje ('53,929184', tekst) na float. Obsluguje
    tez '-' i inne oznaczenia braku danych uzywane w plikach GDDKiA/ZDW."""
    if wartosc is None:
        return None
    if isinstance(wartosc, (int, float)):
        return float(wartosc)
    tekst = str(wartosc).strip()
    if tekst in ("-", "", "brak", "b.d."):
        return None
    try:
        return float(tekst.replace(",", "."))
    except ValueError:
        return None


def wczytaj_dk_geometria(plik):
    wb = openpyxl.load_workbook(plik, read_only=True)
    ws = wb["Zebrane"]
    wiersze = []
    for w in ws.iter_rows(min_row=12, values_only=True):
        if w[0] is None:
            continue
        wiersze.append({
            "odcinek_id": str(w[0]), "droga": w[1],
            "lat_prawy": w[11], "lon_prawy": w[12],
            "lat_lewy": w[15], "lon_lewy": w[16],
            "ten_t": w[26],
        })
    df = pd.DataFrame(wiersze)
    # usredniamy prawy/lewy punkt pomiarowy gdy oba dostepne, w przeciwnym
    # razie uzywamy tego, ktory jest
    df["lat"] = df[["lat_prawy", "lat_lewy"]].mean(axis=1, skipna=True)
    df["lon"] = df[["lon_prawy", "lon_lewy"]].mean(axis=1, skipna=True)
    return df[["odcinek_id", "droga", "lat", "lon", "ten_t"]]


def wczytaj_dk_ruch(plik):
    wb = openpyxl.load_workbook(plik, read_only=True)
    ws = wb["Tab02"]
    wiersze = []
    for w in ws.iter_rows(min_row=11, values_only=True):
        if w[0] is None:
            continue
        wiersze.append({
            "odcinek_id": str(w[0]),
            "sdrr_ogolem": w[8], "sam_osobowe": w[10], "lekkie_ciezarowe": w[12],
        })
    return pd.DataFrame(wiersze)


def wczytaj_dw_geometria(plik):
    wb = openpyxl.load_workbook(plik, read_only=True)
    ws = wb["Zebrane"]
    wiersze = []
    for w in ws.iter_rows(min_row=12, values_only=True):
        if w[0] is None:
            continue
        wiersze.append({
            "odcinek_id": str(w[0]), "droga": w[1],
            "lat": polska_liczba(w[8]), "lon": polska_liczba(w[9]),
        })
    return pd.DataFrame(wiersze)


def wczytaj_dw_ruch(plik):
    wb = openpyxl.load_workbook(plik, read_only=True)
    ws = wb["Tab02"]
    wiersze = []
    for w in ws.iter_rows(min_row=11, values_only=True):
        if w[0] is None:
            continue
        wiersze.append({
            "odcinek_id": str(w[0]),
            "sdrr_ogolem": w[7], "sam_osobowe": w[9], "lekkie_ciezarowe": w[11],
        })
    return pd.DataFrame(wiersze)


def wczytaj_baze_segmentow(plik_geo_dk, plik_ruch_dk, plik_geo_dw, plik_ruch_dw):
    """Wczytuje i laczy wszystkie 4 pliki GPR w jedna, spojna baze
    segmentow pomiarowych (DK + DW razem), gotowa do wyszukiwania
    najblizszego segmentu dla dowolnego punktu."""
    geo_dk = wczytaj_dk_geometria(plik_geo_dk)
    ruch_dk = wczytaj_dk_ruch(plik_ruch_dk)
    dk = geo_dk.merge(ruch_dk, on="odcinek_id", how="inner")
    dk["typ_drogi"] = "dk"

    geo_dw = wczytaj_dw_geometria(plik_geo_dw)
    ruch_dw = wczytaj_dw_ruch(plik_ruch_dw)
    dw = geo_dw.merge(ruch_dw, on="odcinek_id", how="inner")
    dw["typ_drogi"] = "dw"
    dw["ten_t"] = None  # drogi wojewodzkie nie sa czescia sieci TEN-T

    baza = pd.concat([dk, dw], ignore_index=True)
    baza = baza[baza["lat"].notna() & baza["lon"].notna()].copy()

    print(f"Baza segmentow GPR: DK={len(dk)}, DW={len(dw)}, "
          f"razem (z geometria)={len(baza)}")
    return baza


# ============================================================
# PONIZEJ: integracja nowych kandydatow z reszta pipeline'u
# ============================================================

"""
Integruje nowe typy kandydatow (centra handlowe, supermarkety, duze
parkingi - z nowe_typy_osm.csv) z reszta pipeline'u, dopasowujac je do
tego samego schematu co istniejace kandydaty (fuel_station/junction/mop).

Cztery kroki dopasowania:
1. Powiat/TERYT - spatial join z powiaty_teryt.geojson
2. Konkurencja EIPA (1km/2km) - odleglosc do istniejacych stacji EIPA,
   POLICZONA W PELNI (mamy pelna liste stacji w candidate_locations_
   ze_scoringiem.csv)
3. Ruch drogowy - PRAWDZIWE dopasowanie do najblizszego segmentu
   pomiarowego GPR 2025 (drogi krajowe + wojewodzkie), na podstawie
   oficjalnych plikow GDDKiA/ZDW (geometria + natezenie ruchu, patrz
   wczytaj_gpr.py). Identyczna metoda, jaka uzyto dla oryginalnych
   16 232 lokalizacji - NIE jest to juz przyblizenie przez sasiada.
4. Kontekst powiatu (flota EV, populacja, sklonnosc publiczna) -
   dociagniete z istniejacych danych (stale per powiat, weryfikacja:
   max 1 unikalna wartosc w kazdym powiecie).

WAZNE: NIE liczy tu jeszcze udzial_populacji (siatka GUS) - to musi
zrobic gestosc_zaludnienia.ipynb na POLACZONYM (stare+nowe) zbiorze,
zeby mianownik (suma populacji powiatu) pozostal spojny. Ten skrypt
tylko przygotowuje nowe kandydaty do wejscia w ten sam krok.

Wymaga: geopandas, scikit-learn, nowe_typy_osm.csv,
        candidate_locations_ze_scoringiem.csv, powiaty_teryt.geojson
        (wszystkie w ../data/)
Wynik: ../data/candidate_locations_rozszerzone.csv (stare + nowe, gotowe
       do wejscia w gestosc_zaludnienia.ipynb)
"""

import os
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.neighbors import BallTree

# Sciezki do surowych plikow GPR 2025 - podmien na wlasciwe, jesli maja
# inna date w nazwie (data pochodzi z ostatniej aktualizacji GDDKiA/ZDW)
PLIK_GEO_DK = "../data/Tab01_25_dk_KRAJ_20260414.xlsx"
PLIK_RUCH_DK = "../data/Tab02_25_dk_KRAJ_20260414.xlsx"
PLIK_GEO_DW = "../data/Tab01_25_dw_KRAJ_20260622.xlsx"
PLIK_RUCH_DW = "../data/Tab02_25_dw_KRAJ_20260622.xlsx"

# Progi pewnosci dopasowania - te same, co uzywane w model_scoringowy.ipynb
PROG_PEWNOSC_WYSOKA_KM = 2
PROG_PEWNOSC_SREDNIA_KM = 10

PLIK_NOWE = "../data/nowe_typy_osm.csv" if os.path.exists("../data") else "nowe_typy_osm.csv"
PLIK_ISTNIEJACE = "../data/candidate_locations_ze_scoringiem.csv" if os.path.exists("../data") else "candidate_locations_ze_scoringiem.csv"
PLIK_GRANICE = "../data/powiaty_teryt.geojson" if os.path.exists("../data") else "powiaty_teryt.geojson"
PLIK_WYJSCIOWY = "../data/candidate_locations_rozszerzone.csv" if os.path.exists("../data") else "candidate_locations_rozszerzone.csv"

PROMIEN_KONKURENCJI_KM = [1, 2]  # te same promienie co w reszcie modelu
MOC_TYPOWEJ_STACJI_KW = 44.0

# Kolumny stale per powiat - dociagane wprost z istniejacych danych
KOLUMNY_POWIATU = [
    "powiat_nazwa_geo", "powiat_wojewodztwo", "powiat_nazwa",
    "powiat_powierzchnia_km2", "powiat_ludnosc", "powiat_gestosc_zaludnienia",
    "powiat_liczba_bev", "powiat_liczba_phev", "powiat_liczba_ev_razem",
    "powiat_ev_na_1000_mieszkancow", "powiat_liczba_stacji_eipa",
    "powiat_moc_stacji_eipa_kw", "powiat_moc_kw_na_1000_mieszkancow",
    "powiat_scalony_obszar", "powiat_ev_dane_zanizone",
    "sklonnosc_ladowania_publicznego", "udzial_jednorodzinne_nsp",
    # BLAD ZNALEZIONY: brak tej kolumny powodowal NaN w wynik_scoringowy
    # dla WSZYSTKICH nowych kandydatow (energia_kwh_rocznie_szacunek w
    # model_scoringowy.ipynb jest mnozona przez ta kolumne - NaN x cokolwiek
    # = NaN, co dalej wywalalo mape przy proscie zbudowania HeatMap).
    "luka_infrastrukturalna_powiatu", "mnoznik_luki_infrastrukturalnej",
]



def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def main():
    print("Wczytywanie danych...")
    nowe = pd.read_csv(PLIK_NOWE, low_memory=False)

    # ZOSTAWIAMY TYLKO SUPERMARKET: sprawdzilismy geograficzny rozklad
    # wszystkich trzech nowych typow wzgledem prawdziwej wielkosci powiatu
    # (powiat_ludnosc) - centrum_handlowe i parking_duzy sa 2-3x nadreprezen-
    # towane w 10 najwiekszych miastach, a niemal nieobecne w malych
    # miastach (centrum_handlowe: 2,0% zamiast oczekiwanych ~13%). Supermarket
    # ma rozklad niemal identyczny jak stary punkt odniesienia (fuel_station),
    # wiec zostaje sam.
    przed_filtrem = len(nowe)
    nowe = nowe[nowe["source_layer"] == "supermarket"].copy()
    print(f"  Filtr typow: zostawiono {len(nowe)} z {przed_filtrem} "
          f"(odrzucono centrum_handlowe i parking_duzy - nierowny rozklad geograficzny)")
    istniejace = pd.read_csv(PLIK_ISTNIEJACE, low_memory=False)
    istniejace = istniejace[istniejace["dedup_status"] == "unique"].copy()

    # ZABEZPIECZENIE PRZED DUPLIKACJA: jesli PLIK_ISTNIEJACE juz zawiera
    # nowe typy z poprzedniego uruchomienia tego skryptu (np. przy
    # debugowaniu, kiedy caly lancuch uruchamia sie kilka razy pod rzad),
    # usuwamy je PRZED doklejeniem swiezych, zamiast dodawac je ponownie
    # na wierzch. Bez tego kazde kolejne uruchomienie multiplikuje nowe
    # kandydatury (raz doswiadczylismy 3x duplikatu po trzech przebiegach
    # debugowania) - ta linijka czyni skrypt bezpiecznym do wielokrotnego
    # uruchamiania, niezaleznie ile razy juz zostal wczesniej wykonany.
    TYPY_NOWE = ["centrum_handlowe", "supermarket", "parking_duzy"]
    liczba_starych_nowych = istniejace["source_layer"].isin(TYPY_NOWE).sum()
    if liczba_starych_nowych > 0:
        print(f"  Usuwam {liczba_starych_nowych} wierszy nowych typow z poprzedniego "
              f"uruchomienia (zeby nie zduplikowac)")
        istniejace = istniejace[~istniejace["source_layer"].isin(TYPY_NOWE)].copy()

    print(f"  Nowych kandydatow: {len(nowe)}")
    print(f"  Istniejacych lokalizacji (po oczyszczeniu): {len(istniejace)}")

    # ---------- podstawowa struktura nowych kandydatow ----------
    nowe = nowe.rename(columns={"osm_id": "_osm_id"})
    nowe["location_id"] = nowe["source_layer"] + "_" + nowe["_osm_id"].astype(str)
    nowe["segment"] = "docelowa"  # wszystkie nowe typy to lokalizacje docelowe
    nowe["dedup_status"] = "unique"
    nowe["ten_t"] = np.nan  # centra/supermarkety/parkingi nie sa czescia sieci TEN-T

    # ---------- KROK 1: przypisanie do powiatu (spatial join) ----------
    print("\nPrzypisywanie do powiatow (spatial join)...")
    granice = gpd.read_file(PLIK_GRANICE)
    if granice.crs != "EPSG:4326":
        granice = granice.to_crs(epsg=4326)

    nowe_gdf = gpd.GeoDataFrame(
        nowe, geometry=gpd.points_from_xy(nowe.longitude, nowe.latitude), crs="EPSG:4326"
    )
    dopasowane = gpd.sjoin(nowe_gdf, granice[["geometry", "JPT_KOD_JE"]], how="left", predicate="within")
    nowe["teryt_powiat_geo"] = pd.to_numeric(dopasowane["JPT_KOD_JE"], errors="coerce")

    brak_dopasowania = nowe["teryt_powiat_geo"].isna().sum()
    if brak_dopasowania > 0:
        print(f"  UWAGA: {brak_dopasowania} lokalizacji nie dopasowalo powiatu (odrzucam)")
        nowe = nowe[nowe["teryt_powiat_geo"].notna()].copy()
    print(f"  Dopasowano: {len(nowe)}")

    # ---------- KROK 2: konkurencja EIPA (1km / 2km) ----------
    print("\nLiczenie konkurencji EIPA w promieniu 1km/2km...")
    eipa = istniejace[istniejace["source_layer"] == "eipa_station"].copy()
    eipa_aktywne = eipa[eipa["station_suspended"] == False]
    eipa_lat, eipa_lon = eipa_aktywne["latitude"].values, eipa_aktywne["longitude"].values
    eipa_moc = pd.to_numeric(eipa_aktywne["total_power_kw"], errors="coerce").fillna(0).values

    for promien in PROMIEN_KONKURENCJI_KM:
        kolumna = f"existing_eipa_power_kw_active_{promien}km"
        wyniki = []
        for _, wiersz in nowe.iterrows():
            odlegli = haversine_km(wiersz["latitude"], wiersz["longitude"], eipa_lat, eipa_lon)
            wyniki.append(eipa_moc[odlegli <= promien].sum())
        nowe[kolumna] = wyniki
        print(f"  {kolumna}: mediana={pd.Series(wyniki).median():.0f} kW")

    # ---------- KROK 3: ruch drogowy - PRAWDZIWE dopasowanie do najblizszego
    # segmentu pomiarowego GPR 2025 (nie przyblizenie przez sasiada) ----------
    print("\nWczytywanie surowej bazy segmentow pomiarowych GPR 2025...")
    segmenty = wczytaj_baze_segmentow(PLIK_GEO_DK, PLIK_RUCH_DK, PLIK_GEO_DW, PLIK_RUCH_DW)

    # Niektore segmenty maja POPRAWNA geometrie, ale BRAKUJACA/niepoprawna
    # wartosc ruchu (np. "-" lub inny format nieparsowalny na liczbe) -
    # 19 z 5613 w naszej bazie. Bez tego filtra BallTree moglby wybrac taki
    # segment jako "najblizszy" dla jakiegos kandydata, "zarazajac" go NaN
    # w traffic_primary_sam_osobowe, co dalej robilo NaN w calym
    # wynik_scoringowy (bo to iloczyn kilku mnoznikow) i wywalalo mape.
    for kol in ["sam_osobowe", "sdrr_ogolem", "lekkie_ciezarowe"]:
        segmenty[kol] = pd.to_numeric(segmenty[kol], errors="coerce")
    przed_filtrem = len(segmenty)
    segmenty = segmenty[
        segmenty["sam_osobowe"].notna()
        & segmenty["sdrr_ogolem"].notna()
        & segmenty["lekkie_ciezarowe"].notna()
    ].copy()
    if przed_filtrem > len(segmenty):
        print(f"  Odrzucono {przed_filtrem - len(segmenty)} segmentow z niepoprawna wartoscia ruchu")

    print("Dopasowywanie kazdej nowej lokalizacji do najblizszego segmentu...")
    wspolrzedne_segmentow = np.radians(segmenty[["lat", "lon"]].values)
    drzewo_segmentow = BallTree(wspolrzedne_segmentow, metric="haversine")
    wspolrzedne_nowe = np.radians(nowe[["latitude", "longitude"]].values)
    odleglosci_rad, indeksy_najblizszych = drzewo_segmentow.query(wspolrzedne_nowe, k=1)

    R_ZIEMI_KM = 6371.0
    dystans_km = (odleglosci_rad.flatten() * R_ZIEMI_KM)
    dopasowany_segment = segmenty.iloc[indeksy_najblizszych.flatten()].reset_index(drop=True)

    nowe["traffic_primary_dist_m"] = dystans_km * 1000
    nowe["traffic_primary_sam_osobowe"] = pd.to_numeric(dopasowany_segment["sam_osobowe"], errors="coerce").values
    nowe["traffic_primary_sdrr_ogolem"] = pd.to_numeric(dopasowany_segment["sdrr_ogolem"], errors="coerce").values
    nowe["nearest_dk_lekkie_ciezarowe"] = pd.to_numeric(dopasowany_segment["lekkie_ciezarowe"], errors="coerce").values
    nowe["traffic_primary_source"] = dopasowany_segment["typ_drogi"].values
    # UWAGA: CELOWO NIE nadpisujemy tu "ten_t" wartoscia z dopasowanego
    # segmentu. Centrum handlowe/supermarket/parking to NIE sa wezly sieci
    # TEN-T, nawet jesli fizycznie leza blisko takiej drogi - to inny typ
    # obiektu (docelowy, nie korytarzowy). Premia AFIR ma sens dla hubow
    # PRZY trasie, nie dla dowolnego budynku w poblizu niej. "ten_t"
    # pozostaje NaN dla tych typow, tak jak ustawiono na poczatku funkcji.

    nowe["traffic_confidence"] = pd.cut(
        dystans_km,
        bins=[-0.001, PROG_PEWNOSC_WYSOKA_KM, PROG_PEWNOSC_SREDNIA_KM, np.inf],
        labels=["wysoka", "srednia", "niska"],
    ).astype(str)
    nowe["ruch_przyblizony_z_sasiedztwa"] = False  # PRAWDZIWE dopasowanie, nie przyblizenie

    print(f"  Mediana dystansu do najblizszego segmentu: {pd.Series(dystans_km).median():.2f} km")
    print(f"  Rozklad pewnosci dopasowania: {nowe['traffic_confidence'].value_counts().to_dict()}")
    print(f"  Mediana dopasowanego ruchu: {nowe['traffic_primary_sam_osobowe'].median():.0f} sam/dobe")

    # ---------- KROK 4: kontekst powiatu - dociagniety z istniejacych danych ----------
    print("\nDociaganie kontekstu powiatu...")
    # UWAGA: filtrujemy do segmentu 'docelowa' PRZED wyciagnieciem unikalnych
    # wartosci per powiat - kolumny takie jak sklonnosc_ladowania_publicznego
    # sa wypelnione WYLACZNIE dla segmentu docelowego (korytarzowy ich nie
    # uzywa), wiec bez tego filtra drop_duplicates moglby trafic na wiersz
    # korytarzowy i dac pusta wartosc, mimo ze dla tego powiatu istnieje
    # poprawna, niepusta wartosc w wierszach docelowych.
    lookup_powiatu = (
        istniejace[istniejace["segment"] == "docelowa"]
        .drop_duplicates(subset="teryt_powiat_geo")
        .set_index("teryt_powiat_geo")[KOLUMNY_POWIATU]
    )
    nowe = nowe.merge(lookup_powiatu, left_on="teryt_powiat_geo", right_index=True, how="left")

    brak_kontekstu = nowe["powiat_nazwa"].isna().sum()
    if brak_kontekstu > 0:
        print(f"  UWAGA: {brak_kontekstu} lokalizacji bez dopasowanego kontekstu powiatu (odrzucam)")
        nowe = nowe[nowe["powiat_nazwa"].notna()].copy()

    # ---------- polaczenie z istniejacymi danymi ----------
    print("\nLaczenie z istniejacymi danymi...")
    wspolne_kolumny = [k for k in istniejace.columns if k in nowe.columns]
    brakujace_w_nowych = [k for k in istniejace.columns if k not in nowe.columns]
    for k in brakujace_w_nowych:
        nowe[k] = np.nan

    polaczone = pd.concat([istniejace, nowe[istniejace.columns]], ignore_index=True)
    polaczone.to_csv(PLIK_WYJSCIOWY, index=False)

    print(f"\n{'='*60}")
    print(f"Zapisano: {PLIK_WYJSCIOWY}")
    print(f"  Bylo: {len(istniejace)}, dodano: {len(nowe)}, razem: {len(polaczone)}")
    print(f"\nKOLEJNY KROK: uruchom gestosc_zaludnienia.ipynb na tym pliku")
    print(f"(zmien PLIK_WEJSCIOWY na '{PLIK_WYJSCIOWY}'), zeby przeliczyc")
    print(f"udzial_populacji na PELNYM, rozszerzonym zbiorze kandydatow.")


if __name__ == "__main__":
    main()


Wczytywanie danych...
  Filtr typow: zostawiono 9401 z 11192 (odrzucono centrum_handlowe i parking_duzy - nierowny rozklad geograficzny)
  Usuwam 11192 wierszy nowych typow z poprzedniego uruchomienia (zeby nie zduplikowac)
  Nowych kandydatow: 9401
  Istniejacych lokalizacji (po oczyszczeniu): 14520

Przypisywanie do powiatow (spatial join)...
  Dopasowano: 9401

Liczenie konkurencji EIPA w promieniu 1km/2km...
  existing_eipa_power_kw_active_1km: mediana=44 kW
  existing_eipa_power_kw_active_2km: mediana=230 kW

Wczytywanie surowej bazy segmentow pomiarowych GPR 2025...


c:\Users\mateu\anaconda3\envs\python11\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


Baza segmentow GPR: DK=2424, DW=3305, razem (z geometria)=5613
  Odrzucono 19 segmentow z niepoprawna wartoscia ruchu
Dopasowywanie kazdej nowej lokalizacji do najblizszego segmentu...
  Mediana dystansu do najblizszego segmentu: 1.85 km
  Rozklad pewnosci dopasowania: {'wysoka': 4919, 'srednia': 4397, 'niska': 85}
  Mediana dopasowanego ruchu: 9119 sam/dobe

Dociaganie kontekstu powiatu...

Laczenie z istniejacymi danymi...

Zapisano: ../data/candidate_locations_rozszerzone.csv
  Bylo: 14520, dodano: 9401, razem: 23921

KOLEJNY KROK: uruchom gestosc_zaludnienia.ipynb na tym pliku
(zmien PLIK_WEJSCIOWY na '../data/candidate_locations_rozszerzone.csv'), zeby przeliczyc
udzial_populacji na PELNYM, rozszerzonym zbiorze kandydatow.
